# Paper #48 — Allende Prieto (2016): Solar and Stellar Photospheric Abundances

## Implementation Notebook / 구현 노트북

This notebook reproduces the foundational calculations behind modern photospheric abundance analysis as reviewed by Allende Prieto (2016): the curve of growth, Saha ionization balance, Voigt line profile, Eddington–Barbier emergent intensity, LTE vs NLTE departure schematic, and the Grevesse–Sauval (1998) vs Asplund et al. (2009) solar abundance comparison.

본 노트북은 Allende Prieto(2016)가 리뷰한 현대 광구 조성 분석의 기반 계산을 재현한다: 성장곡선, Saha 이온화 균형, Voigt 선 프로파일, Eddington–Barbier 방출 강도, LTE 대 NLTE 이탈 개념도, Grevesse–Sauval(1998) 대 Asplund 등(2009) 태양 조성 비교.

In [ ]:
"""Setup and imports.

Standard scientific Python stack plus physical constants.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import wofz

# Physical constants (SI) / 물리 상수 (SI 단위)
H_PLANCK = 6.62607015e-34        # J s
C_LIGHT = 2.99792458e8           # m / s
K_B = 1.380649e-23               # J / K
ME = 9.1093837015e-31            # kg
EV = 1.602176634e-19             # J per eV

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print("Environment ready / 환경 준비 완료.")

## 1. Curve of Growth / 성장곡선

The curve of growth relates the logarithmic equivalent width $\log(W_\lambda/\lambda)$ to the logarithm of the product $N f \lambda$ (column density × oscillator strength × wavelength). Three regimes emerge:

1. **Weak-line (linear)**: $W_\lambda \propto N f$ — abundance determination ideal.
2. **Saturation (flat)**: $W_\lambda \propto \sqrt{\ln(Nf)}$ — weak dependence on N.
3. **Damping (strong-line)**: $W_\lambda \propto \sqrt{N f \Gamma}$ — Lorentzian wings dominate.

성장곡선은 $\log(W_\lambda/\lambda)$ 대 $\log(Nf\lambda)$ 관계로, (1) 약한 선 선형, (2) 포화, (3) 감쇠 영역이 나타난다.

In [ ]:
def curve_of_growth(log_Nf_lambda: np.ndarray, a: float = 1e-3) -> np.ndarray:
    """Compute the schematic curve of growth.

    Three asymptotic regimes connected by smooth transitions; the analytic
    forms follow Gray (2005) "The Observation and Analysis of Stellar
    Photospheres" (chap. 16).

    Args:
        log_Nf_lambda: Logarithm of the product N * f * lambda (dimensionless).
        a: Voigt damping parameter controlling onset of the strong-line regime.

    Returns:
        log10 of W_lambda / lambda as an array of the same shape.
    """
    x = 10 ** log_Nf_lambda
    # Weak-line linear
    W_linear = 1e-6 * x
    # Saturation (Gaussian core, Doppler-broadened)
    W_saturation = 1e-5 * np.sqrt(np.log1p(x * 1e-4))
    # Damping (Lorentzian wings)
    W_damping = 3e-7 * np.sqrt(a * x * 1e-2)
    W_total = np.maximum.reduce([W_linear, W_saturation, W_damping])
    return np.log10(W_total + 1e-20)


log_Nf = np.linspace(0, 12, 400)
log_W = curve_of_growth(log_Nf)

fig, ax = plt.subplots()
ax.plot(log_Nf, log_W, color="navy", linewidth=2)
ax.axvspan(0, 4, color="#a2d5f2", alpha=0.3, label="Linear / 선형")
ax.axvspan(4, 8, color="#ffd166", alpha=0.3, label="Saturation / 포화")
ax.axvspan(8, 12, color="#ef476f", alpha=0.3, label="Damping / 감쇠")
ax.set_xlabel(r"$\log(N f \lambda)$")
ax.set_ylabel(r"$\log(W_\lambda / \lambda)$")
ax.set_title("Curve of Growth / 성장곡선")
ax.legend(loc="lower right")
plt.show()

## 2. Saha Ionization for Fe, O, Na / Fe, O, Na의 Saha 이온화

The Saha equation
$$ \frac{N_{j+1} n_e}{N_j} = \frac{2 u_{j+1}}{u_j} \frac{(2\pi m_e k T)^{3/2}}{h^3} e^{-\chi_j/kT} $$
gives ionization fractions at a fixed electron density $n_e$.

We compute the neutral fraction $N_I / (N_I + N_{II})$ vs temperature for Fe (χ=7.90 eV), O (χ=13.62 eV), Na (χ=5.14 eV) at a solar-photospheric $n_e \sim 10^{13}\,\mathrm{cm^{-3}}$.

Saha 방정식으로 T에 대한 중성 분율을 Fe, O, Na에 대해 계산한다.

In [ ]:
def saha_neutral_fraction(T: np.ndarray, chi_eV: float, n_e_cgs: float = 1e13,
                          u_ratio: float = 1.0) -> np.ndarray:
    """Compute the neutral fraction N_I / (N_I + N_II) via Saha's equation.

    Args:
        T: Temperature in kelvin.
        chi_eV: First ionization potential in eV.
        n_e_cgs: Electron number density in cm^-3. Converted to SI internally.
        u_ratio: 2 * u_{II} / u_{I} partition-function ratio (assumed ~1).

    Returns:
        Neutral fraction array with the same shape as T.
    """
    n_e_SI = n_e_cgs * 1e6  # cm^-3 -> m^-3
    chi_J = chi_eV * EV
    thermal = (2 * np.pi * ME * K_B * T) ** 1.5 / H_PLANCK ** 3
    ratio = u_ratio * thermal / n_e_SI * np.exp(-chi_J / (K_B * T))
    return 1.0 / (1.0 + ratio)


T_grid = np.linspace(3000, 12000, 400)
elements = {
    "Fe": (7.902, "tab:orange"),
    "O":  (13.618, "tab:blue"),
    "Na": (5.139, "tab:red"),
}

fig, ax = plt.subplots()
for name, (chi, color) in elements.items():
    frac = saha_neutral_fraction(T_grid, chi)
    ax.plot(T_grid, frac, label=f"{name} (χ={chi} eV)", color=color, linewidth=2)
ax.axvline(5778, color="gray", linestyle="--", alpha=0.7, label="Solar T_eff")
ax.set_xlabel("Temperature / 온도 (K)")
ax.set_ylabel(r"Neutral fraction $N_I / (N_I + N_{II})$")
ax.set_title("Saha Ionization vs Temperature / 온도에 따른 Saha 이온화")
ax.legend()
plt.show()

**Interpretation / 해석**. At T ≈ 5778 K, oxygen is essentially 100% neutral (χ=13.62 eV is far from kT ≈ 0.5 eV), iron is ~50%–80% ionized into Fe II (χ=7.90 eV), and sodium is almost fully ionized into Na II (χ=5.14 eV). Hence: the solar O I 777 nm triplet and Fe I lines sample the dwindling neutral tails for their species; Na I D lines come from a very small neutral minority.

태양 T_eff에서 O는 거의 100% 중성, Fe는 상당 부분 1가 이온화, Na는 거의 전부 이온화. 따라서 Fe I·Na I·O I 흡수선은 각기 다른 집단의 "소수파"에서 비롯된다.

## 3. Synthetic Weak Absorption Line / 합성 약한 흡수선

A Voigt profile convolves a Doppler Gaussian (thermal + microturbulence) with a Lorentzian (natural + collisional damping):

$$ \phi(\nu) = \frac{\mathrm{Re}\,[w(z)]}{\sqrt{\pi}\,\Delta\nu_D}, \quad z = \frac{(\nu - \nu_0) + i\Gamma/(4\pi)}{\Delta\nu_D} $$

We reproduce a weak Fe I line near λ = 6000 Å with modest damping and compare against a Gaussian-only approximation.

Voigt 프로파일을 사용해 6000 Å 근방의 약한 Fe I 선을 합성하고, 감쇠 날개가 없는 단순 Gaussian과 비교한다.

In [ ]:
def voigt_profile(wavelength: np.ndarray, lam0: float, doppler_width: float,
                  gamma: float) -> np.ndarray:
    """Compute a normalized Voigt profile in wavelength units.

    Args:
        wavelength: Wavelength array in angstrom.
        lam0: Line center wavelength in angstrom.
        doppler_width: Doppler width Δλ_D in angstrom.
        gamma: Lorentzian HWHM in angstrom.

    Returns:
        Profile values (per angstrom).
    """
    u = (wavelength - lam0) / doppler_width
    a = gamma / doppler_width
    z = u + 1j * a
    return np.real(wofz(z)) / (np.sqrt(np.pi) * doppler_width)


def gaussian_profile(wavelength: np.ndarray, lam0: float,
                     doppler_width: float) -> np.ndarray:
    """Compute a Gaussian-only profile for comparison."""
    u = (wavelength - lam0) / doppler_width
    return np.exp(-u ** 2) / (np.sqrt(np.pi) * doppler_width)


lam0 = 6000.0          # Å
doppler = 0.05         # Å (thermal + microturbulence for T~5800 K, Fe, ξ=1 km/s)
gamma_damp = 0.02      # Å (Lorentzian damping)
optical_depth_0 = 0.5  # weak line

lam = np.linspace(5999.0, 6001.0, 2000)
phi_voigt = voigt_profile(lam, lam0, doppler, gamma_damp)
phi_gauss = gaussian_profile(lam, lam0, doppler)

# Residual flux = exp(-tau_0 * phi / phi_center)
tau_voigt = optical_depth_0 * phi_voigt / phi_voigt.max()
tau_gauss = optical_depth_0 * phi_gauss / phi_gauss.max()
flux_voigt = np.exp(-tau_voigt)
flux_gauss = np.exp(-tau_gauss)

fig, ax = plt.subplots()
ax.plot(lam, flux_gauss, color="tab:blue", linestyle="--",
        label="Gaussian only / 가우시안만")
ax.plot(lam, flux_voigt, color="tab:red",
        label="Voigt (Gaussian + Lorentzian) / Voigt")
ax.axhline(1.0, color="gray", alpha=0.5)
ax.set_xlabel(r"Wavelength $\lambda$ (Å)")
ax.set_ylabel(r"Normalized flux $F_\lambda/F_c$")
ax.set_title("Weak Fe I Line — Voigt vs Gaussian / 약한 Fe I 선: Voigt 대 가우시안")
ax.legend()
plt.show()

W_voigt = np.trapz(1 - flux_voigt, lam) * 1000  # mÅ
W_gauss = np.trapz(1 - flux_gauss, lam) * 1000
print(f"W_lambda (Voigt)    = {W_voigt:.2f} mÅ")
print(f"W_lambda (Gaussian) = {W_gauss:.2f} mÅ")

## 4. Eddington–Barbier Approximation / Eddington–Barbier 근사

For a plane-parallel LTE atmosphere, the emergent intensity at inclination $\mu = \cos\theta$ is approximately
$$ I_\nu(\mu) \approx S_\nu(\tau_\nu = \mu). $$
At disk center (μ=1) the emergent intensity samples the source function at τ=1; toward the limb (μ→0) progressively shallower layers are sampled — a key reason *center-to-limb* variations test NLTE (Figure 5 of Allende Prieto).

We model a linear source function $S_\nu(\tau) = a + b\tau$ and compute $I(\mu)$ both analytically and by numerical integration of the formal solution.

Eddington–Barbier 근사: 방출 강도는 $\tau = \mu$에서의 원천 함수와 같다. 선형 원천 함수 $S = a + b\tau$를 가정하고 해석·수치 해를 비교한다.

In [ ]:
def eddington_barbier(mu: np.ndarray, a: float, b: float) -> np.ndarray:
    """Eddington–Barbier emergent intensity for a linear source function.

    Args:
        mu: cos(theta), inclination cosine.
        a: Constant term of S(tau) = a + b*tau.
        b: Slope of the source function.

    Returns:
        Approximate I(mu) = a + b * mu.
    """
    return a + b * mu


def formal_solution_linear_S(mu: np.ndarray, a: float, b: float) -> np.ndarray:
    """Exact emergent intensity for a linear source function.

    I(mu) = integral_0^inf S(tau) exp(-tau/mu) dtau/mu,
    which for S = a + b*tau gives I(mu) = a + b*mu.
    The integral evaluates to the same expression for a linear S, so we
    demonstrate the integral numerically on a tau grid.
    """
    tau = np.linspace(0, 20, 2000)
    results = np.empty_like(mu)
    for i, m in enumerate(mu):
        S = a + b * tau
        integrand = S * np.exp(-tau / m) / m
        results[i] = np.trapz(integrand, tau)
    return results


mu = np.linspace(0.05, 1.0, 40)
a_src, b_src = 1.0, 0.7  # arbitrary normalized source function
I_eb = eddington_barbier(mu, a_src, b_src)
I_exact = formal_solution_linear_S(mu, a_src, b_src)

fig, ax = plt.subplots()
ax.plot(mu, I_exact, color="black", linewidth=2,
        label="Exact formal solution / 정확한 형식해")
ax.plot(mu, I_eb, "o", color="tab:red",
        label=r"Eddington–Barbier $I(\mu) = S(\tau = \mu)$")
ax.set_xlabel(r"$\mu = \cos\theta$")
ax.set_ylabel(r"Emergent intensity $I(\mu)$")
ax.set_title("Eddington–Barbier vs Exact (linear S) / Eddington–Barbier 대 정확 해")
ax.invert_xaxis()  # disk center (mu=1) on left
ax.legend()
plt.show()

**Physical meaning / 물리적 의미**. The agreement shown confirms that for a linear source function, $I(\mu) = S(\tau=\mu)$ is exact. Real atmospheres have curved $S(\tau)$; the Eddington–Barbier relation remains accurate to first order and is the conceptual backbone of center-to-limb diagnostics of NLTE, as used in Allende Prieto et al. (2004a) for the O I 777 nm triplet.

선형 원천 함수에서 Eddington–Barbier는 정확하다. 실제 대기의 곡률 때문에 1차 근사로 쓰이며, O I 777 nm 삼중선의 중심–가장자리 NLTE 진단의 개념적 근간이다.

## 5. LTE vs NLTE Departure Schematic / LTE 대 NLTE 이탈 개념도

Define departure coefficients $b_i = n_i / n_i^{LTE}$. Under LTE, $b_i \equiv 1$ at all depths. In the outer atmosphere, level populations over-populate or under-populate relative to LTE because radiative rates couple them to the non-local radiation field.

We mimic a typical Fe I scenario: the ground level stays near LTE in deep layers but becomes *overpopulated* by recombination cascades near the surface, while an excited level is *underpopulated* by photon losses (the classical Fe I over-ionization in metal-poor stars).

LTE에서는 이탈 계수 $b_i \equiv 1$이지만 NLTE에서는 외부 대기에서 복사장 결합으로 값이 1에서 벗어난다. 금속 결핍 별의 Fe I 과이온화 시나리오를 본뜬 개념도를 그린다.

In [ ]:
log_tau = np.linspace(-5, 1, 400)

# LTE reference
b_lte = np.ones_like(log_tau)

# Schematic NLTE departures (illustrative)
b_ground = 1.0 + 0.5 / (1.0 + np.exp(2 * (log_tau + 0.5)))
b_upper = 1.0 - 0.6 / (1.0 + np.exp(2 * (log_tau + 0.5)))

fig, ax = plt.subplots()
ax.plot(log_tau, b_lte, color="gray", linestyle="--",
        label=r"LTE: $b_i = 1$")
ax.plot(log_tau, b_ground, color="tab:blue",
        label=r"Ground level (overpopulated) / 바닥 준위")
ax.plot(log_tau, b_upper, color="tab:red",
        label=r"Excited level (underpopulated) / 여기 준위")
ax.axvspan(-3, 0, alpha=0.1, color="green",
           label="Line-forming region / 선 형성 영역")
ax.set_xlabel(r"$\log \tau_{\rm Ross}$")
ax.set_ylabel(r"Departure coefficient $b_i = n_i/n_i^{\rm LTE}$")
ax.set_title("LTE vs NLTE Departure Coefficients (schematic) / 이탈 계수 개념도")
ax.legend()
plt.show()

In low-metallicity 1D Fe I analyses, this pattern produces abundance corrections of +0.3 to +0.6 dex; in 3D models they can reach +0.9 dex (Shchukina et al. 2005 for HD 140283, cited in Allende Prieto 2016 p. 13).

저금속도 별의 1D Fe I 분석에서는 +0.3–+0.6 dex, 3D에서는 +0.9 dex에 달하는 조성 보정을 만든다(Shchukina et al. 2005; HD 140283).

## 6. Solar Abundance Comparison: Grevesse–Sauval 1998 vs Asplund et al. 2009

Compare the classical Grevesse–Sauval 1998 (GS98) solar composition to the revised Asplund et al. 2009 (AGSS09) values for key elements. The bulk of the reduction is in C, N, O (~40–50% lower), producing the *solar abundance problem*.

Grevesse–Sauval 1998(GS98)과 Asplund 등 2009(AGSS09) 태양 조성을 주요 원소에 대해 비교한다. C, N, O에서 40–50%의 감소가 "태양 조성 문제"를 일으켰다.

In [ ]:
elements = ["C", "N", "O", "Ne", "Na", "Mg", "Si", "Fe"]
A_GS98 =  [8.52, 7.92, 8.83, 8.08, 6.33, 7.58, 7.55, 7.50]
A_AGSS = [8.43, 7.83, 8.69, 7.93, 6.24, 7.60, 7.51, 7.50]

x = np.arange(len(elements))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width / 2, A_GS98, width, color="tab:blue",
       label="Grevesse & Sauval 1998")
ax.bar(x + width / 2, A_AGSS, width, color="tab:red",
       label="Asplund et al. 2009 (AGSS09)")
ax.set_xticks(x)
ax.set_xticklabels(elements)
ax.set_ylabel(r"$A(X) = \log(N_X/N_H) + 12$")
ax.set_title("Solar Photospheric Abundances: GS98 vs AGSS09 / 태양 광구 조성 비교")
ax.legend()
for i, (gs, asp) in enumerate(zip(A_GS98, A_AGSS)):
    delta = asp - gs
    ax.annotate(f"{delta:+.2f}", xy=(i, max(gs, asp) + 0.1),
                ha="center", fontsize=9,
                color="red" if delta < 0 else "black")
ax.set_ylim(6.0, 9.3)
plt.tight_layout()
plt.show()

# Compute Z/X for each composition (approximate — heavy elements only)
def z_over_x(abundances: dict, atomic_mass: dict) -> float:
    """Approximate Z/X from A(X) values.

    Only includes the listed metals and ignores He. Sufficient for an
    illustrative comparison.
    """
    N_H = 1.0  # normalization
    numerator = 0.0
    for elem, A in abundances.items():
        n_rel = 10 ** (A - 12)  # N_X / N_H
        numerator += n_rel * atomic_mass[elem]
    return numerator / (N_H * 1.008)

atomic_mass = {"C": 12.011, "N": 14.007, "O": 15.999, "Ne": 20.180,
               "Na": 22.990, "Mg": 24.305, "Si": 28.086, "Fe": 55.845}

zx_GS98 = z_over_x(dict(zip(elements, A_GS98)), atomic_mass)
zx_AGSS = z_over_x(dict(zip(elements, A_AGSS)), atomic_mass)
print(f"Approximate Z/X (GS98 partial) = {zx_GS98:.4f}")
print(f"Approximate Z/X (AGSS09 partial) = {zx_AGSS:.4f}")
print(f"Full Z/X quoted in literature: GS98 ≈ 0.0231, AGSS09 ≈ 0.0181")
print(f"Relative reduction: {(1 - zx_AGSS / zx_GS98) * 100:.1f}%")

**The solar abundance problem / 태양 조성 문제**.

The reduction in the Asplund scale (especially for O and C) drops the global Z/X from ≈ 0.023 (GS98) to ≈ 0.018 (AGSS09). Standard solar-interior models with AGSS09 composition disagree with helioseismic inversions that constrain:
- the radius of the convection-zone base $r_{\rm BCZ}/R_\odot \approx 0.7133 \pm 0.0005$ (from p-mode frequency ratios),
- the surface helium mass fraction $Y_{\rm surf} \approx 0.2485 \pm 0.0035$.

The Bailey et al. (2015) Z-pinch laboratory measurement of Fe interior opacity came out significantly higher than theoretical Opacity-Project predictions, potentially alleviating the tension without reverting to GS98.

Asplund 척도는 Z/X를 0.023에서 0.018로 낮춰 헬리오사이스몰로지가 주는 대류층 바닥 반경 $r_{\rm BCZ}/R_\odot \approx 0.7133$과 표면 헬륨 질량 분율 $Y_{\rm surf} \approx 0.2485$와 충돌한다. Bailey 등(2015)의 Fe 불투명도 실험이 해결책이 될 수 있다.

## 7. Summary / 요약

This notebook reproduced six foundational calculations behind stellar photospheric abundance analysis as reviewed by Allende Prieto (2016):

1. **Curve of growth** — three regimes (linear, saturation, damping), demonstrating why weak lines are the abundance workhorse.
2. **Saha ionization** — showing why O is neutral, Fe is partially ionized, and Na is nearly fully ionized in the solar photosphere.
3. **Voigt profile** — thermal Doppler core + Lorentzian damping wings on a weak Fe I line.
4. **Eddington–Barbier approximation** — $I(\mu) = S(\tau=\mu)$ as the formal basis of center-to-limb NLTE diagnostics.
5. **LTE vs NLTE departure coefficients** — schematic of Fe I over-ionization in the outer atmosphere.
6. **Solar abundance comparison** — Grevesse–Sauval 1998 vs Asplund et al. 2009, with the consequential Z/X reduction driving the solar abundance problem.

본 노트북은 (1) 성장곡선, (2) Saha 이온화, (3) Voigt 프로파일, (4) Eddington–Barbier 근사, (5) LTE 대 NLTE 이탈 계수, (6) 태양 조성 비교 — 총 6개의 기초 계산을 재현했다. 모든 결과가 Allende Prieto(2016) 리뷰의 주요 개념을 정량적으로 뒷받침한다.